In [1]:
import numpy as np

class GridWorld:
    def __init__(self):
        self.rows, self.cols = 3, 4
        self.wall = (1, 1)
        self.terminals = {(0,3): 1, (1,3): -1}
        self.actions = ["U","D","L","R"]
        self.gamma = 0.9
        self.step = -0.01

    def is_terminal(self, s):
        return s in self.terminals

    def move(self, s, a):
        if self.is_terminal(s): return s
        r,c = s
        if a=="U": r-=1
        if a=="D": r+=1
        if a=="L": c-=1
        if a=="R": c+=1
        ns=(r,c)
        if r<0 or r>=self.rows or c<0 or c>=self.cols or ns==self.wall:
            return s
        return ns


def policy_evaluation(env, policy, V, theta=1e-5):
    while True:
        delta = 0
        V_old = V.copy()

        for r in range(env.rows):
            for c in range(env.cols):
                s = (r,c)

                if s == env.wall: continue
                if env.is_terminal(s):
                    V[r,c] = env.terminals[s]
                    continue

                a = policy[r,c]
                ns = env.move(s,a)

                if env.is_terminal(ns):
                    v = env.step + env.gamma * env.terminals[ns]
                else:
                    nr,nc = ns
                    v = env.step + env.gamma * V_old[nr,nc]

                delta = max(delta, abs(v - V[r,c]))
                V[r,c] = v

        if delta < theta:
            break

    return V


def policy_improvement(env, V, policy):
    stable = True

    for r in range(env.rows):
        for c in range(env.cols):
            s = (r,c)

            if s == env.wall or env.is_terminal(s):
                continue

            old_action = policy[r,c]
            best_action, best_value = None, -1e9

            for a in env.actions:
                ns = env.move(s,a)

                if env.is_terminal(ns):
                    v = env.terminals[ns]
                else:
                    nr,nc = ns
                    v = env.step + env.gamma * V[nr,nc]

                if v > best_value:
                    best_value = v
                    best_action = a

            policy[r,c] = best_action

            if old_action != best_action:
                stable = False

    return policy, stable


def policy_iteration(env):
    V = np.zeros((env.rows, env.cols))
    policy = np.random.choice(env.actions, (env.rows, env.cols))

    while True:
        V = policy_evaluation(env, policy, V)
        policy, stable = policy_improvement(env, V, policy)

        if stable:
            break

    return V, policy


env = GridWorld()
V, policy = policy_iteration(env)

print("Value Function:\n", V)
print("\nPolicy:\n", policy)

Value Function:
 [[ 0.7019    0.791     0.89      1.      ]
 [ 0.62171   0.        0.791    -1.      ]
 [ 0.549539  0.62171   0.7019    0.62171 ]]

Policy:
 [['R' 'R' 'R' 'L']
 ['U' 'L' 'U' 'R']
 ['U' 'R' 'U' 'L']]
